# Chapter 12: MLOps

## 12.4 ML Pipeline

In [ ]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, accuracy_score, roc_auc_score
from sklearn.impute import SimpleImputer

: 

In [ ]:
# Description about the data set





In [ ]:
chd_df = pd.read_csv("SAheart.data")

In [4]:
chd_df.columns

Index(['row.names', 'sbp', 'tobacco', 'ldl', 'adiposity', 'famhist', 'typea',
       'obesity', 'alcohol', 'age', 'chd'],
      dtype='object')

In [ ]:

#sbp:	Systolic blood pressure(Numerical)
#tobacco:	Cumulative tobacco consumption(Numerical)
#ldl:	Low-density lipoprotein cholesterol(Numerical)
#adiposity:	Measure of body fat/adiposity(Numerical)
#famhist:	Family history of heart disease	(Categorical}
#typea:	Type-A behavior score	(Numerical)
#obesity:	Body-mass index/obesity measure	(Numerical)
#alcohol:	Current alcohol consumption	(Numerical)
#age:	Age of the person	(Numerical)
#chd:	Whether coronary heart disease is present	(Binary response)

In [5]:
chd_df.head(5)

,row.names,sbp,tobacco,ldl,adiposity,famhist,typea,obesity,alcohol,age,chd
0,1,160,12.00,5.73,23.11,Present,49,25.30,97.20,52,1
1,2,144,0.01,4.41,28.61,Absent,55,28.87,2.06,63,1
2,3,118,0.08,3.48,32.28,Present,52,29.14,3.81,46,0
3,4,170,7.50,6.41,38.03,Present,51,31.99,24.26,58,1
4,5,134,13.60,3.50,27.78,Present,60,25.99,57.34,49,1


In [ ]:
# y is our response variable, X is set of predictor variables.
# Our main goal is to predict the chd_df(y) based on the predictors variables(X).

In [6]:
X = chd_df.drop('chd', axis=1)
y = chd_df['chd']

In [7]:
X_train, X_test, y_train, y_test = train_test_split(X,
                                                    y,
                                                    test_size=0.2,
                                                    random_state=42)

In [8]:
categorical_features = ['famhist']
numerical_features = ['sbp',
                      'tobacco',
                      'ldl',
                      'adiposity',
                      'typea',
                      'obesity',
                      'alcohol',
                      'age']

In [9]:
categorical_features

['famhist']

In [10]:
numerical_features

['sbp', 'tobacco', 'ldl', 'adiposity', 'typea', 'obesity', 'alcohol', 'age']

In [11]:
numerical_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='mean')),
    ('scaler', StandardScaler())
])

In [12]:
categorical_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('onehot', OneHotEncoder(handle_unknown='ignore'))
])

In [13]:
preprocessor = ColumnTransformer(
    transformers=[
        ('num', numerical_transformer, numerical_features),
        ('cat', categorical_transformer, categorical_features)
    ]
)

In [14]:
pipeline = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('classifier', LogisticRegression(max_iter=1000))
])

In [15]:
pipeline

,steps,"[('preprocessor', ...), ('classifier', ...)]"
,transform_input,None
,memory,None
,verbose,False
,transformers,"[('num', ...), ('cat', ...)]"
,remainder,'drop'
,sparse_threshold,0.3
,n_jobs,None
,transformer_weights,None
,verbose,False
,verbose_feature_names_out,True


In [16]:
pipeline.fit(X_train, y_train)

,steps,"[('preprocessor', ...), ('classifier', ...)]"
,transform_input,None
,memory,None
,verbose,False
,transformers,"[('num', ...), ('cat', ...)]"
,remainder,'drop'
,sparse_threshold,0.3
,n_jobs,None
,transformer_weights,None
,verbose,False
,verbose_feature_names_out,True


In [17]:
y_pred = pipeline.predict(X_test)

In [18]:
print(classification_report(y_test, y_pred))

              precision    recall  f1-score   support

           0       0.78      0.92      0.84        59
           1       0.79      0.56      0.66        34

    accuracy                           0.78        93
   macro avg       0.79      0.74      0.75        93
weighted avg       0.79      0.78      0.77        93



In [19]:
from joblib import dump
dump(pipeline, 'chd.pickle')

['chd.pickle']

In [21]:
#%pip install mlflow


     ---------------------------------------- 11.2/11.2 MB 3.4 MB/s eta 0:00:00
     ---------------------------------------- 2.2/2.2 MB 4.3 MB/s eta 0:00:00
     ---------------------------------------- 56.2/56.2 KB 2.9 MB/s eta 0:00:00
     -------------------------------------- 114.9/114.9 KB 3.4 MB/s eta 0:00:00
     -------------------------------------- 148.8/148.8 KB 4.3 MB/s eta 0:00:00
     ---------------------------------------- 3.8/3.8 MB 3.1 MB/s eta 0:00:00
     ---------------------------------------- 3.6/3.6 MB 3.3 MB/s eta 0:00:00
     ---------------------------------------- 27.8/27.8 MB 3.1 MB/s eta 0:00:00
     -------------------------------------- 132.2/132.2 KB 1.9 MB/s eta 0:00:00
     -------------------------------------- 123.9/123.9 KB 2.4 MB/s eta 0:00:00
     -------------------------------------- 265.9/265.9 KB 2.0 MB/s eta 0:00:00
     -------------------------------------- 480.5/480.5 KB 3.8 MB/s eta 0:00:00
     -------------------------------------- 10

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
ortools 9.11.4210 requires protobuf<5.27,>=5.26.1, but you have protobuf 6.33.6 which is incompatible.
You should consider upgrading via the 'C:\Users\Dell\AppData\Local\Programs\Python\Python310\python.exe -m pip install --upgrade pip' command.


In [20]:
import mlflow
from mlflow.models import infer_signature

In [21]:
# Set our tracking server uri for logging
mlflow.set_tracking_uri(uri="http://127.0.0.1:8080")

# Create a new MLflow Experiment
mlflow.set_experiment("CHD_Prediction")

# Start an MLflow run
with mlflow.start_run():
    # Log the loss metric
    mlflow.log_metric("roc", np.round(roc_auc_score(y_test, y_pred), 3))

    # Set a tag that we can use to remind ourselves what this run was for
    mlflow.set_tag("Training", "Logistic Regression")

    # Infer the model signature
    signature = infer_signature(X_train,
                                pipeline.predict(X_train))

    # Log the model
    model_info = mlflow.sklearn.log_model(
        sk_model=pipeline,
        artifact_path="logreg",
        signature=signature,
        input_example=X_train,
        registered_model_name="logistic",
    )

MlflowException: API request to http://127.0.0.1:8080/api/2.0/mlflow/experiments/get-by-name failed with exception HTTPConnectionPool(host='127.0.0.1', port=8080): Max retries exceeded with url: /api/2.0/mlflow/experiments/get-by-name?experiment_name=CHD_Prediction (Caused by NewConnectionError('<urllib3.connection.HTTPConnection object at 0x000001F4FBFABCD0>: Failed to establish a new connection: [WinError 10061] No connection could be made because the target machine actively refused it'))